## 반복부호(Repetition code) 오류 정정 실습 프로그램.

### 개요:
- 입력 데이터 비트열의 각 비트를 r번 반복하여 전송한다. (r = 3,5,7,... 권장)
- 수신 측은 각 블록(r개)에서 다수결(majority vote)로 원래 비트를 추정한다.

### 실습 포인트:
- r이 홀수면 동률(tie)이 없어서 디코딩이 단순해진다.
- 한 블록에서 (r-1)/2개 이하의 비트가 뒤집히면 다수결로 정정 가능하다.
- (r+1)/2개 이상이 뒤집히면 오정정(miscorrection)될 수 있다.

### 오류 주입 방식:
1) 수동: 전체 코드워드에서 뒤집을 비트 위치(1부터 시작)를 직접 입력
2) 확률: 비트 반전 확률 p(0~1)를 입력하면 난수로 뒤집기 (시드 설정 가능)
"""

In [ ]:

from __future__ import annotations

import random


def validate_bits(bits: str) -> bool:
    return len(bits) > 0 and all(ch in {"0", "1"} for ch in bits)


def flip_bit(bit: str) -> str:
    return "1" if bit == "0" else "0"


def repetition_encode(data_bits: str, r: int) -> str:
    return "".join(bit * r for bit in data_bits)


def majority_decode_block(block: str) -> str:
    ones = block.count("1")
    zeros = len(block) - ones
    # r이 홀수면 동률이 없다. r이 짝수면 동률 시 '1'로 결정(일관성 위한 규칙).
    if ones > zeros:
        return "1"
    if zeros > ones:
        return "0"
    return "1"


def repetition_decode(codeword: str, r: int) -> tuple[str, list[tuple[int, str, str]]]:
    """
    returns:
      - decoded_bits
      - per_block_info: [(block_index_1_based, received_block, decided_bit), ...]
    """
    decoded: list[str] = []
    info: list[tuple[int, str, str]] = []
    for i in range(0, len(codeword), r):
        block = codeword[i : i + r]
        decided = majority_decode_block(block)
        decoded.append(decided)
        info.append((i // r + 1, block, decided))
    return "".join(decoded), info


def parse_positions(raw: str, max_len: int) -> list[int]:
    if not raw.strip():
        return []
    tokens = raw.replace(",", " ").split()
    positions: list[int] = []
    for t in tokens:
        if not t.isdigit():
            raise ValueError(f"숫자가 아닌 입력: {t}")
        value = int(t)
        if value < 1 or value > max_len:
            raise ValueError(f"범위를 벗어난 위치: {value} (허용: 1~{max_len})")
        positions.append(value)
    return positions


def inject_errors_manual(codeword: str, positions_1_based: list[int]) -> tuple[str, list[int]]:
    bits = list(codeword)
    flipped: list[int] = []
    for pos in positions_1_based:
        idx = pos - 1
        bits[idx] = flip_bit(bits[idx])
        flipped.append(pos)
    return "".join(bits), flipped


def inject_errors_prob(codeword: str, p: float, seed: int | None) -> tuple[str, list[int]]:
    if seed is not None:
        random.seed(seed)
    bits = list(codeword)
    flipped_positions: list[int] = []
    for i in range(len(bits)):
        if random.random() < p:
            bits[i] = flip_bit(bits[i])
            flipped_positions.append(i + 1)  # 1-based
    return "".join(bits), flipped_positions


def count_bit_errors(a: str, b: str) -> int:
    return sum(1 for x, y in zip(a, b, strict=True) if x != y)


def run_interactive() -> None:
    print("=" * 60)
    print("반복부호(Repetition code) 오류 정정 실습")
    print("=" * 60)

    while True:
        data_bits = input("\n데이터 비트열 입력(0/1): ").strip()
        if validate_bits(data_bits):
            break
        print("입력 오류: 0과 1로만 구성된 비트열을 입력하세요.")

    while True:
        r_raw = input("반복 횟수 r 입력(예: 3, 5, 7...): ").strip()
        if r_raw.isdigit() and int(r_raw) >= 2:
            r = int(r_raw)
            break
        print("입력 오류: 2 이상의 정수를 입력하세요.")

    if r % 2 == 0:
        print("주의: r이 짝수이면 동률(tie) 가능성이 있어, 동률 시 '1'로 결정하는 규칙을 사용합니다.")

    codeword = repetition_encode(data_bits, r)
    print("\n[송신 측]")
    print(f"- 원본 데이터: {data_bits}")
    print(f"- 반복 횟수 r: {r}")
    print(f"- 코드워드 길이: {len(codeword)} (={len(data_bits)}*{r})")
    print(f"- 생성 코드워드: {codeword}")

    print("\n[전송 중 오류 주입 방식 선택]")
    print("1) 수동(특정 위치 비트 반전)")
    print("2) 확률(각 비트가 p로 반전)")

    while True:
        method = input("선택(1/2): ").strip()
        if method in {"1", "2"}:
            break
        print("입력 오류: 1 또는 2를 입력하세요.")

    if method == "1":
        print(f"\n- 위치는 1부터 시작, 범위는 1~{len(codeword)}")
        print("- 예: 3 또는 2,5 또는 1 4 7")
        print("- 엔터만 누르면 오류 없이 전송")
        while True:
            raw = input("비트 반전 위치 입력: ").strip()
            try:
                positions = parse_positions(raw, len(codeword))
                received, flipped_positions = inject_errors_manual(codeword, positions)
                break
            except ValueError as ex:
                print(f"입력 오류: {ex}")
    else:
        while True:
            p_raw = input("\n비트 반전 확률 p 입력(0~1, 예: 0.05): ").strip()
            try:
                p = float(p_raw)
            except ValueError:
                print("입력 오류: 실수로 입력하세요.")
                continue
            if 0.0 <= p <= 1.0:
                break
            print("입력 오류: 0 이상 1 이하의 값을 입력하세요.")

        seed_raw = input("난수 시드(seed) 입력(재현용, 엔터=랜덤): ").strip()
        seed = int(seed_raw) if seed_raw else None
        received, flipped_positions = inject_errors_prob(codeword, p, seed)

    print("\n[수신 측]")
    print(f"- 수신 코드워드: {received}")

    decoded, per_block = repetition_decode(received, r)
    print(f"- 디코딩 결과(추정 데이터): {decoded}")

    channel_errors = count_bit_errors(codeword, received)
    residual_errors = count_bit_errors(data_bits, decoded)

    print("\n[결과 요약]")
    print(f"- 전송 중 뒤집힌 비트 수: {channel_errors}")
    print(f"- 정정 후 남은 데이터 비트 오류 수: {residual_errors}")
    if residual_errors == 0:
        print("- 결론: 오류를 모두 정정했습니다.")
    else:
        print("- 결론: 일부 블록에서 다수결이 깨져 오정정/미정정이 발생했습니다.")

    if flipped_positions:
        head = flipped_positions[:30]
        tail = "" if len(flipped_positions) <= 30 else f" ... (총 {len(flipped_positions)}개)"
        print(f"\n참고: 실제 주입된 오류 위치(일부) = {head}{tail}")

    show_detail = input("\n블록별(수신 r비트, 다수결) 상세를 볼까요? (y/N): ").strip().lower()
    if show_detail == "y":
        print("\n[블록별 상세]")
        for idx, block, decided in per_block:
            print(f"- 블록 {idx:>3}: {block} -> {decided}")


if __name__ == "__main__":
    run_interactive()
